 Movie Recommendations HW
 Name: Srikala Sandu

**Collaboration Policy:** Homeworks will be done individually: each student must hand in their own answers. Use of partial or entire solutions obtained from others or online is strictly prohibited.


**Late Policy:** Late submission have a penalty of 2\% for each passing hour.


**Submission format:** Successfully complete the Movie Lens recommender as described in this jupyter notebook. Submit a `.py` and an `.ipynb` file for this notebook. You can go to `File -> Download as ->` to download a .py version of the notebook.

**Only submit one `.ipynb` file and one `.py` file.** The `.ipynb` file should have answers to all the questions. Do *not* zip any files for submission.

**## Reading the Data**

In [43]:
column_list_ratings = ["UserID", "MovieID", "Ratings","Timestamp"]
ratings_data = pd.read_csv(
    'ratings.dat',
    sep='::',
    names=column_list_ratings,
    engine='python'
)


In [42]:
import numpy as np
import pandas as pd


In [46]:
from google.colab import files
uploaded = files.upload()


Saving users.dat to users.dat


In [50]:
from google.colab import files
uploaded = files.upload()


Saving users.dat to users (1).dat


**## Data Analysis**
We now have all our data in Pandas - however, it's as three separate datasets! To make some more sense out of the data we have, we can use the Pandas `merge` function to combine our component data-frames. Run the following code:

In [52]:
from google.colab import files
uploaded = files.upload()

Saving users.dat to users (2).dat


In [53]:
import pandas as pd

column_list_users = ["UserID","Gender","Age","Occupation","Zip-code"]

user_data = pd.read_csv(
    "users.dat",       # Make sure the file is in the current directory
    sep='::',
    names=column_list_users,
    engine='python'
)

user_data.head()  # check the first few rows


,UserID,Gender,Age,Occupation,Zip-code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [54]:
# Merge ratings with user data on 'UserID'
ratings_users = pd.merge(ratings_data, user_data, on='UserID')

# Merge the result with movies data on 'MovieID'
data = pd.merge(ratings_users, movies_data, on='MovieID')

data.head()  # show first few rows


,UserID,MovieID,Ratings,Timestamp,Gender,Age,Occupation,Zip-code,Title,Genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy


Let’s create the pivot table using different variable names so it’s clearer and avoids conflicts. For example, we can use movie_ratings for the pivot table.

In [55]:
# Create a pivot table to get the average rating for each movie
movie_ratings = data.pivot_table(
    index='Title',      # Movies as rows
    values='Ratings',   # Ratings to aggregate
    aggfunc='mean'      # Take the average
)

# Display the first few rows
movie_ratings.head()


,Ratings
Title,
"$1,000,000 Duck (1971)",3.027027
'Night Mother (1986),3.371429
'Til There Was You (1997),2.692308
"'burbs, The (1989)",2.910891
...And Justice for All (1979),3.713568


To combine both average rating and number of ratings, we can create two pivot tables (or use agg to do both at once. This will help see which movies are not only highly rated but also widely reviewed.

In [56]:
# Create a pivot table with both average rating and number of ratings
movie_stats = data.groupby('Title').agg(
    average_rating=('Ratings', 'mean'),
    rating_count=('Ratings', 'count')
)

# Display the first few rows
movie_stats.head()


,average_rating,rating_count
Title,,
"$1,000,000 Duck (1971)",3.027027,37
'Night Mother (1986),3.371429,70
'Til There Was You (1997),2.692308,52
"'burbs, The (1989)",2.910891,303
...And Justice for All (1979),3.713568,199


movie_stats – contains average rating and number of ratings for each movie:

In [57]:
movie_stats = data.groupby('Title').agg(
    average_rating=('Ratings', 'mean'),
    rating_count=('Ratings', 'count')
)
movie_stats.head()


,average_rating,rating_count
Title,,
"$1,000,000 Duck (1971)",3.027027,37
'Night Mother (1986),3.371429,70
'Til There Was You (1997),2.692308,52
"'burbs, The (1989)",2.910891,303
...And Justice for All (1979),3.713568,199


top_15_mean_ratings – shows top 15 movies purely by average rating:

In [58]:
mean_ratings = data.pivot_table('Ratings', index=["Title"], aggfunc='mean')
top_15_mean_ratings = mean_ratings.sort_values(by='Ratings', ascending=False).head(15)
top_15_mean_ratings


,Ratings
Title,
Lured (1947),5.000000
Ulysses (Ulisse) (1954),5.000000
"Baby, The (1973)",5.000000
Follow the Bitch (1998),5.000000
Schlafes Bruder (Brother of Sleep) (1995),5.000000
Bittersweet Motel (2000),5.000000
Song of Freedom (1936),5.000000
"Gate of Heavenly Peace, The (1995)",5.000000
One Little Indian (1973),5.000000


Update the original `mean_ratings` function to include a `columns` parameter that calculates separate average ratings for men and women.


In [59]:
# Pivot table to show average rating by movie and gender
gender_ratings = data.pivot_table(
    values='Ratings',     # Ratings to aggregate
    index='Title',        # Movies as rows
    columns='Gender',     # Separate columns for each gender
    aggfunc='mean'        # Take the average
)

# Display the first few rows
gender_ratings.head()


Gender,F,M
Title,,
"$1,000,000 Duck (1971)",3.375000,2.761905
'Night Mother (1986),3.388889,3.352941
'Til There Was You (1997),2.675676,2.733333
"'burbs, The (1989)",2.793478,2.962085
...And Justice for All (1979),3.828571,3.689024


We can now sort ratings by the `F` and `M` columns instead of `Rating` and print the top movies based on female and male reviews, respectively.


In [62]:
data = pd.merge(pd.merge(ratings_data, user_data, on='UserID'), movies_data, on='MovieID')
mean_ratings = data.pivot_table(
    'Ratings',
    index=["Title"],
    columns=["Gender"],
    aggfunc='mean'
)
top_female_ratings = mean_ratings.sort_values(by='F', ascending=False)
print(top_female_ratings.head(15))

top_male_ratings = mean_ratings.sort_values(by='M', ascending=False)
print(top_male_ratings.head(15))



Gender                                                F         M
Title                                                            
Coldblooded (1995)                                  5.0  3.588235
Clean Slate (Coup de Torchon) (1981)                5.0  3.857143
Dancemaker (1998)                                   5.0  3.250000
Skipped Parts (2000)                                5.0  4.000000
Country Life (1994)                                 5.0  2.000000
Lamerica (1994)                                     5.0  4.666667
Message to Love: The Isle of Wight Festival (1996)  5.0  3.833333
Ayn Rand: A Sense of Life (1997)                    5.0  4.000000
Twice Upon a Yesterday (1998)                       5.0  3.222222
Ballad of Narayama, The (Narayama Bushiko) (1958)   5.0  3.428571
Saltmen of Tibet, The (1997)                        5.0  3.571429
Raw Deal (1948)                                     5.0  3.307692
Brother, Can You Spare a Dime? (1975)               5.0  3.642857
Song of Fr

In [63]:
# Calculate difference between male and female ratings
mean_ratings['diff'] = mean_ratings['M'] - mean_ratings['F']

# Sort by difference
sorted_by_diff = mean_ratings.sort_values(by='diff')

# Top 10 movies where females rated higher than males
sorted_by_diff[:10]


Gender,F,M,diff
Title,,,
"Spiders, The (Die Spinnen, 1. Teil: Der Goldene See) (1919)",4.000000,1.000000,-3.000000
"James Dean Story, The (1957)",4.000000,1.000000,-3.000000
Country Life (1994),5.000000,2.000000,-3.000000
Babyfever (1994),3.666667,1.000000,-2.666667
"Woman of Paris, A (1923)",5.000000,2.428571,-2.571429
Cobra (1925),4.000000,1.500000,-2.500000
"Other Side of Sunday, The (Søndagsengler) (1996)",5.000000,2.928571,-2.071429
Theodore Rex (1995),3.000000,1.000000,-2.000000
"Separation, The (La Séparation) (1994)",4.000000,2.000000,-2.000000


I group the DataFrame by Title to compare the number of ratings. Then, select the top 10 titles by review count to see the most reviewed movies.

In [64]:
# Count number of ratings per movie
ratings_by_title = data.groupby('Title').size()

top_10_most_rated = ratings_by_title.sort_values(ascending=False).head(10)

top_10_most_rated

,0
Title,
American Beauty (1999),3428
Star Wars: Episode IV - A New Hope (1977),2991
Star Wars: Episode V - The Empire Strikes Back (1980),2990
Star Wars: Episode VI - Return of the Jedi (1983),2883
Jurassic Park (1993),2672
Saving Private Ryan (1998),2653
Terminator 2: Judgment Day (1991),2649
"Matrix, The (1999)",2590
Back to the Future (1985),2583


## Question 1
Create a ratings matrix using Numpy. This matrix allows us to see the ratings for a given movie and user ID. The element at location $[i,j]$ is a rating given by user $i$ for movie $j$. Print the **shape** of the matrix produced.  

Additionally, choose 3 users that have rated the movie with MovieID "**1377**" (Batman Returns). Print these ratings, they will be used later for comparison.


**Notes: (READ CAREFULLY)**
- Do *not* use `pivot_table`.
- A ratings matrix is *not* the same as `ratings_data` from above.
- The ratings of movie with MovieID $i$ are stored in the ($i$-1)th column (index starts from 0)  
- Not every user has rated every movie. Missing entries should be set to 0 for now.
- If you're stuck, you might want to look into `np.zeros` and how to use it to create a matrix of the desired shape.
- Every review lies between 1 and 5.

## Question 2
Normalize the ratings matrix (created in **Question 1**) using Z-score normalization. While we can't use `sklearn`'s `StandardScaler` for this step, we can do the statistical calculations ourselves to normalize the data.

Before you start:
- Your first step should be to get the average of every *column* of the ratings matrix (we want an average by title, not by user!).
- Make sure that the mean is calculated considering only non-zero elements. If there is a movie which is rated only by 10 users, we get its mean rating using (sum of the 10 ratings)/10 and **NOT** (sum of 10 ratings)/(total number of users)
- All of the missing values in the dataset should be replaced with the average rating for the given movie. This is a complex topic, but for our case replacing empty values with the mean will make it so that the absence of a rating doesn't affect the overall average, and it provides an "expected value" which is useful for computing correlations and recommendations in later steps.
- In our matrix, 0 represents a missing rating.
- Next, we want to subtract the average from the original ratings thus allowing us to get a mean of 0 in every *column*. It may be very close but not exactly zero because of the limited precision `float`s allow.
- Lastly, divide this by the standard deviation of the *column*.

- Not every MovieID is used, leading to zero columns. This will cause a divide by zero error when normalizing the matrix. Simply replace any NaN values in your normalized matrix with 0.

In [14]:
import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()

ratings_data = pd.read_csv(
    'ratings.dat',
    sep='::',
    engine='python',
    names=['UserID', 'MovieID', 'Rating', 'Timestamp']
)


print(ratings_data.head())


Saving ratings.dat to ratings (4).dat
   UserID  MovieID  Rating  Timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291


In [15]:
# Number of users and movies
total_viewers = ratings_data['UserID'].max()
total_titles = ratings_data['MovieID'].max()

# Initialize the ratings matrix
score_matrix = np.zeros((total_viewers, total_titles))

# Fill the matrix
for entry in ratings_data.itertuples():
    score_matrix[entry[1]-1, entry[2]-1] = entry[3]

# Print matrix shape
print("Shape of Ratings Matrix:", score_matrix.shape)

# MovieID for Batman Returns
batman_movie_id = 1377
viewers_who_rated = np.where(score_matrix[:, batman_movie_id-1] > 0)[0][:3]

print("\nUsers who rated Batman Returns (MovieID 1377):", viewers_who_rated)

# Print ratings
for viewer in viewers_who_rated:
    print(f"User {viewer+1} rating for Batman Returns:", score_matrix[viewer, batman_movie_id-1])


Shape of Ratings Matrix: (6040, 3952)

Users who rated Batman Returns (MovieID 1377): [ 9 12 17]
User 10 rating for Batman Returns: 3.0
User 13 rating for Batman Returns: 3.0
User 18 rating for Batman Returns: 2.0


In [17]:
import numpy as np

# Step 1: Copy original ratings matrix
zscore_matrix = score_matrix.copy().astype(float)

# Step 2: Compute mean rating per movie (column), ignoring zeros
num_viewers, num_titles = zscore_matrix.shape
title_means = np.zeros(num_titles)

for col_idx in range(num_titles):
    non_zero_ratings = zscore_matrix[:, col_idx][zscore_matrix[:, col_idx] > 0]
    if len(non_zero_ratings) > 0:
        title_means[col_idx] = non_zero_ratings.mean()
    else:
        title_means[col_idx] = 0

# Step 3: Replace missing ratings (zeros) with the movie mean
for col_idx in range(num_titles):
    missing_indices = zscore_matrix[:, col_idx] == 0
    zscore_matrix[missing_indices, col_idx] = title_means[col_idx]

# Step 4: Subtract mean from each column
for col_idx in range(num_titles):
    zscore_matrix[:, col_idx] -= title_means[col_idx]

# Step 5: Divide by standard deviation of each column
title_std = np.std(zscore_matrix, axis=0, ddof=0)
title_std[title_std == 0] = 1  # avoid divide by zer_]()_


In [18]:
# Number of viewers (users)
num_viewers = score_matrix.shape[0]

# Initialize arrays
viewer_means = np.zeros(num_viewers)
user_normalized_matrix = score_matrix.copy()

# Normalize ratings per user (subtract user's mean)
for v_idx in range(num_viewers):
    rated_mask = user_normalized_matrix[v_idx, :] > 0
    if rated_mask.any():
        mean_rating = user_normalized_matrix[v_idx, rated_mask].mean()
        viewer_means[v_idx] = mean_rating
        user_normalized_matrix[v_idx, rated_mask] -= mean_rating

# Display first 10 users' mean ratings
print("Viewer means (first 10):", viewer_means[:10])


Viewer means (first 10): [4.18867925 3.71317829 3.90196078 4.19047619 3.14646465 3.90140845
 4.32258065 3.88489209 3.73584906 4.11471322]


## Question 3
We're now going to perform Singular Value Decomposition (SVD) on the normalized ratings matrix from the previous question. Perform the process using numpy, and along the way print the shapes of the $U$, $S$, and $V$ matrices you calculated.

In [19]:
import numpy as np

# Perform Singular Value Decomposition (SVD) on the user-normalized ratings matrix
U_matrix, singular_values, Vt_matrix = np.linalg.svd(user_normalized_matrix, full_matrices=False)

# Print the shapes of the decomposed matrices
print("U_matrix shape:", U_matrix.shape)
print("Singular values vector shape:", singular_values.shape)
print("Vt_matrix shape:", Vt_matrix.shape)


U_matrix shape: (6040, 3952)
Singular values vector shape: (3952,)
Vt_matrix shape: (3952, 3952)


## Question 4
Reconstruct four rank-k rating matrix $R_k$, where $R_k = U_kS_kV_k^T$ for k = [100, 1000, 2000, 3000]. Using each of $R_k$ make predictions for the 3 users selected in Question 1, for the movie with ID 1377 (Batman Returns). Compare the original ratings with the predicted ratings.

In [20]:
# Define different k values for low-rank approximation
k_values = [2, 5, 10, 50]

# Store reconstructed matrices
reconstructed_matrices = {}

# Reconstruct the ratings matrix for each k
for k in k_values:
    U_k = U_matrix[:, :k]
    S_k = np.diag(singular_values[:k])
    Vt_k = Vt_matrix[:k, :]

    # Reconstruct normalized ratings
    Rk_norm = U_k @ S_k @ Vt_k

    # Add back the user means to get actual predicted ratings
    Rk_predicted = Rk_norm + viewer_means.reshape(-1, 1)

    reconstructed_matrices[k] = Rk_predicted

# Compare actual vs predicted ratings for Batman Returns (MovieID 1377)
print("Predicted vs actual ratings for Batman Returns (MovieID 1377):")
for viewer in viewers_who_rated:
    actual_rating = score_matrix[viewer, batman_movie_id-1]
    print(f"\nUser {viewer+1}, actual rating: {actual_rating:.1f}")
    for k in k_values:
        predicted_rating = reconstructed_matrices[k][viewer, batman_movie_id-1]
        print(f"  k={k} prediction: {predicted_rating:.2f}")


Predicted vs actual ratings for Batman Returns (MovieID 1377):

User 10, actual rating: 3.0
  k=2 prediction: 4.08
  k=5 prediction: 4.07
  k=10 prediction: 4.02
  k=50 prediction: 3.89

User 13, actual rating: 3.0
  k=2 prediction: 3.31
  k=5 prediction: 3.21
  k=10 prediction: 3.21
  k=50 prediction: 3.29

User 18, actual rating: 2.0
  k=2 prediction: 3.38
  k=5 prediction: 3.43
  k=10 prediction: 3.25
  k=50 prediction: 2.62


## Question 5
### Cosine Similarity
Cosine similarity is a metric used to measure how similar two vectors are. Mathematically, it measures the cosine of the angle between two vectors projected in a multi-dimensional space. Cosine similarity is high if the angle between two vectors is 0, and the output value ranges within $cosine(x,y) \in [0,1]$. $0$ means there is no similarity (perpendicular), where $1$ (parallel) means that both the items are 100% similar.

$$ cosine(x,y) = \frac{x^T y}{||x|| ||y||}  $$
**Based on the reconstruction rank-1000 rating matrix $R_{1000}$ and the cosine similarity,** sort the movies which are most similar. You will have a function `top_movie_similarity` which sorts data by its similarity to a movie with ID `movie_id` and returns the top $n$ items, and a second function `print_similar_movies` which prints the titles of said similar movies. Return the top 5 movies for the movie with ID `1377` (*Batman Returns*)

Note: While finding the cosine similarity, there are a few empty columns which will have a magnitude of **zero** resulting in NaN values. These should be replaced by 0, otherwise these columns will show most similarity with the given movie.

In [25]:
from google.colab import files

uploaded = files.upload()  # This will open a file chooser


Saving movies.dat to movies.dat


In [27]:
import pandas as pd

movies_data = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",  # handles special characters
    names=["MovieID", "Title", "Genres"]
)

movies_data.head()


,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [37]:
# Reconstruct the rank-1000 normalized matrix
rank_k1000 = 1000
U_k1000_matrix = U_matrix[:, :rank_k1000]
S_k1000_values = singular_values[:rank_k1000]
Vt_k1000_matrix = Vt_matrix[:rank_k1000, :]
S_k1000_matrix = np.diag(S_k1000_values)
R_k1000_normalized = U_k1000_matrix @ S_k1000_matrix @ Vt_k1000_matrix

def get_top_movie_similarity(R_mat, target_movie_id, top_n=5):
    """
    R_mat: normalized, reconstructed ratings matrix (User x Movie)
    target_movie_id: MovieID to find similarity for
    """
    col_idx = target_movie_id - 1
    target_vector = R_mat[:, col_idx]

    # Compute L2 norm of target movie
    norm_target = np.linalg.norm(target_vector)

    # Precompute norms of all movie vectors
    movie_norms = np.linalg.norm(R_mat, axis=0)

    similarity_list = []
    num_movies = R_mat.shape[1]

    for j in range(num_movies):
        if j == col_idx:
            continue  # skip the movie itself

        current_vector = R_mat[:, j]
        norm_current = movie_norms[j]

        # Cosine similarity
        dot_product = target_vector.T @ current_vector
        if norm_target == 0 or norm_current == 0:
            similarity = 0.0
        else:
            similarity = dot_product / (norm_target * norm_current)

        similarity_list.append((j + 1, similarity))  # (MovieID, similarity)

    # Sort by similarity descending
    similarity_list.sort(key=lambda x: x[1], reverse=True)

    # Return top N movie IDs
    return [movie_id for movie_id, sim in similarity_list[:top_n]]

def show_similar_movies(movie_df, movie_ids):
    print("Most similar movies:")
    for mid in movie_ids:
        title = movie_df[movie_df['MovieID'] == mid]['Title'].iloc[0]
        print(f"- {title} (MovieID: {mid})")

# Find and display top 5 similar movies for Batman Returns
batman_mid = 1377
top5_movies = get_top_movie_similarity(R_k1000_normalized, batman_mid, top_n=5)
show_similar_movies(movies_data, top5_movies)





Most similar movies:
- Batman Forever (1995) (MovieID: 153)
- Dick Tracy (1990) (MovieID: 2616)
- Batman & Robin (1997) (MovieID: 1562)
- Waterworld (1995) (MovieID: 208)
- Single White Female (1992) (MovieID: 3274)


## Question 6
### Movie Recommendations
Using the same process from Question 5, write `top_user_similarity` which sorts data by its similarity to a user with ID `user_id` and returns the top result. Then find the MovieIDs of the movies that this similar user has rated most highly, but that `user_id` has not yet seen. Find at least 5 movie recommendations for the user with ID `5954` and print their titles.

Hint: To check your results, find the genres of the movies that the user likes and compare with the genres of the recommended movies.

In [41]:
import numpy as np
import pandas as pd

# -------------------------------
# 1. User-User Cosine Similarity
# -------------------------------
def get_most_similar_user(R_mat, target_user_id):
    """
    R_mat: normalized, reconstructed ratings matrix (User x Movie)
    target_user_id: UserID to find most similar user for
    Returns: UserID of the most similar user
    """
    row_idx = target_user_id - 1
    target_vector = R_mat[row_idx, :]

    norm_target = np.linalg.norm(target_vector)
    user_norms = np.linalg.norm(R_mat, axis=1)

    similarity_list = []
    num_users = R_mat.shape[0]

    for i in range(num_users):
        if i == row_idx:
            continue

        current_vector = R_mat[i, :]
        norm_current = user_norms[i]

        dot_product = target_vector @ current_vector.T
        if norm_target == 0 or norm_current == 0:
            similarity = 0.0
        else:
            similarity = dot_product / (norm_target * norm_current)

        similarity_list.append((i + 1, similarity))  # (UserID, similarity)

    # Sort descending and return most similar user
    similarity_list.sort(key=lambda x: x[1], reverse=True)
    return similarity_list[0][0]

# -------------------------------
# 2. Recommend Movies
# -------------------------------
def recommend_movies_from_similar_user(ratings_df, movies_df, target_user_id, similar_user_id, min_rating=4, top_n=5):
    """
    ratings_df: original ratings dataframe
    movies_df: movies dataframe
    target_user_id: UserID to recommend for
    similar_user_id: most similar user
    min_rating: threshold for highly rated movies
    top_n: number of recommendations
    Returns: list of top recommended MovieIDs
    """
    # Get highly rated movies from similar user
    similar_user_ratings = ratings_df[ratings_df['UserID'] == similar_user_id]
    highly_rated_movies = similar_user_ratings[similar_user_ratings['Rating'] >= min_rating]

    # Exclude movies already seen by the target user
    target_user_movies = ratings_df[ratings_df['UserID'] == target_user_id]['MovieID'].unique()
    recommended_movies_df = highly_rated_movies[~highly_rated_movies['MovieID'].isin(target_user_movies)]

    # Sort by rating descending and pick top N unique MovieIDs
    top_movie_ids = recommended_movies_df.sort_values(by='Rating', ascending=False)['MovieID'].unique()[:top_n]

    return top_movie_ids

# -------------------------------
# 3. Display Recommendations
# -------------------------------
def show_movie_recommendations(movies_df, movie_ids):
    print("Top Recommended Movies:")
    for mid in movie_ids:
        title = movies_df[movies_df['MovieID'] == mid]['Title'].iloc[0]
        genre = movies_df[movies_df['MovieID'] == mid]['Genres'].iloc[0]
        print(f"- {title} (Genres: {genre})")

# -------------------------------
# 4. Run Recommendations
# -------------------------------
# Make sure your normalized rank-1000 SVD matrix is R_k1000_normalized
target_user = 5954
most_similar_user = get_most_similar_user(R_k1000_normalized, target_user)

print(f"Target User ID: {target_user}")
print(f"Most Similar User ID: {most_similar_user}\n")

top5_recommended_movie_ids = recommend_movies_from_similar_user(
    ratings_data,
    movies_data,
    target_user_id=target_user,
    similar_user_id=most_similar_user,
    min_rating=4,
    top_n=5
)

show_movie_recommendations(movies_data, top5_recommended_movie_ids)


Target User ID: 5954
Most Similar User ID: 5367

Top Recommended Movies:
- Princess Mononoke, The (Mononoke Hime) (1997) (Genres: Action|Adventure|Animation)
- Grosse Pointe Blank (1997) (Genres: Comedy|Crime)
- American Beauty (1999) (Genres: Comedy|Drama)
- Payback (1999) (Genres: Action|Thriller)
- Heavenly Creatures (1994) (Genres: Drama|Fantasy|Romance|Thriller)
